# Mini-project 2: Speech-to-Text Transcription

Self-contained notebook: a small transcription tool built on `SpeechRecognition`, scored on three public clips.
Runtime: under a minute. Needs internet on the first run (to fetch the clips). Part of the [ml-mini-projects](README.md) set.

In [1]:
import os, json, time, random, urllib.request, urllib.parse, urllib.error
from pathlib import Path

import numpy as np

SEED = 42
random.seed(SEED); np.random.seed(SEED)

ROOT = Path.cwd()
DATA = ROOT / "data"
DATA.mkdir(exist_ok=True)

RESULTS = {}   # collected metrics -> saved as JSON at the end
GH = "https://raw.githubusercontent.com/"

def download(url, dest, retries=3):
    """Download url to dest (skipped if the file is already cached)."""
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    for attempt in range(retries):
        try:
            with urllib.request.urlopen(url, timeout=60) as r:
                payload = r.read()
            tmp = dest.with_suffix(dest.suffix + ".part")
            tmp.write_bytes(payload)
            tmp.replace(dest)
            return dest
        except Exception:
            if attempt == retries - 1:
                raise
            time.sleep(2 * (attempt + 1))


# Speech-to-Text Transcription (`SpeechRecognition`)

The `SpeechRecognition` library is a thin wrapper that gives one interface to several recognition engines:

| Engine | Needs internet? | Notes |
|---|---|---|
| `recognize_google` | yes | Google's free Web Speech API. Best quality, but a web service |
| `recognize_sphinx` | **no** | CMU PocketSphinx. Runs locally, noticeably less accurate |

The tool below transcribes audio files with either engine. To measure it honestly, three public speech clips with
known transcripts are used and the **word error rate (WER)** is computed:
`WER = (substitutions + insertions + deletions) / words in the reference` (lower is better, 0 is perfect).

Clips: a ~11 s John F. Kennedy excerpt (from the OpenAI Whisper test files), a short "go forward ten meters"
command (PocketSphinx test data), and a "one two three" clip (SpeechRecognition's examples).

In [2]:
import soundfile as sf
from scipy.signal import resample_poly
import speech_recognition as sr

AUDIO = DATA / "audio"
GH = "https://raw.githubusercontent.com/"
CLIPS = {
    "jfk": (GH + "openai/whisper/main/tests/jfk.flac", "jfk.flac",
            "and so my fellow americans ask not what your country can do for you ask what you can do for your country"),
    "goforward": (GH + "cmusphinx/pocketsphinx/master/test/data/goforward.raw", "goforward.raw",
                  "go forward ten meters"),
    "english": (GH + "Uberi/speech_recognition/master/examples/english.wav", "english.wav",
                "one two three"),
}

def prepare_wav(src, dst, target_rate=16000):
    """Convert any clip to mono 16 kHz 16-bit WAV with a proper low-pass resampler."""
    src = Path(src)
    if src.suffix == ".raw":   # headerless 16 kHz / 16-bit / mono PCM
        audio, rate = sf.read(src, samplerate=16000, channels=1, format="RAW", subtype="PCM_16")
    else:
        audio, rate = sf.read(src)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if rate != target_rate:
        g = np.gcd(int(rate), target_rate)
        audio = resample_poly(audio, target_rate // g, int(rate) // g)
    sf.write(dst, np.clip(audio, -1, 1), target_rate, subtype="PCM_16")
    return dst

wavs = {}
for name, (url, fname, ref) in CLIPS.items():
    raw_path = download(url, AUDIO / fname)
    wavs[name] = prepare_wav(raw_path, AUDIO / f"{name}_16k.wav")
    info = sf.info(wavs[name])
    print(f"{name:10s} {info.duration:5.1f}s  {info.samplerate} Hz  channels={info.channels}")

jfk         11.0s  16000 Hz  channels=1
goforward    2.8s  16000 Hz  channels=1
english      2.7s  16000 Hz  channels=1


In [3]:
recognizer = sr.Recognizer()

def transcribe(wav_path, engine="sphinx", language="en-US"):
    """Transcribe an audio file. Returns '' if no speech was understood."""
    with sr.AudioFile(str(wav_path)) as source:
        audio = recognizer.record(source)
    try:
        if engine == "sphinx":
            return recognizer.recognize_sphinx(audio, language=language)
        if engine == "google":
            return recognizer.recognize_google(audio, language=language)
        raise ValueError(f"unknown engine {engine!r}")
    except sr.UnknownValueError:
        return ""

def normalize(text):
    return " ".join("".join(c if c.isalnum() or c in " '" else " " for c in text.lower()).split())

def wer(reference, hypothesis):
    """Word error rate via word-level edit distance."""
    r, h = normalize(reference).split(), normalize(hypothesis).split()
    d = np.zeros((len(r) + 1, len(h) + 1), dtype=int)
    d[:, 0] = range(len(r) + 1); d[0, :] = range(len(h) + 1)
    for i in range(1, len(r) + 1):
        for j in range(1, len(h) + 1):
            d[i, j] = min(d[i - 1, j] + 1, d[i, j - 1] + 1, d[i - 1, j - 1] + (r[i - 1] != h[j - 1]))
    return d[-1, -1] / len(r)

assert wer("a b c", "a b c") == 0 and abs(wer("a b c d", "a x c") - 0.5) < 1e-9   # quick sanity test

In [4]:
TRY_GOOGLE = True       # set False to skip the online engine
google_ok = None
speech_rows = []
for name, (_, _, ref) in CLIPS.items():
    row = {"clip": name, "seconds": round(sf.info(wavs[name]).duration, 1), "reference": ref}
    row["sphinx"] = transcribe(wavs[name], "sphinx")
    row["sphinx_wer"] = round(wer(ref, row["sphinx"]), 3)
    if TRY_GOOGLE and google_ok is not False:
        try:
            row["google"] = transcribe(wavs[name], "google")
            row["google_wer"] = round(wer(ref, row["google"]), 3)
            google_ok = True
        except sr.RequestError as e:
            google_ok = False
            print("Google Web Speech API not reachable from this machine:", e, "\n")
    speech_rows.append(row)

for row in speech_rows:
    print(f"[{row['clip']}] ({row['seconds']}s)")
    print("  reference :", row["reference"])
    print("  sphinx    :", row["sphinx"], f"  -> WER {row['sphinx_wer']:.0%}")
    if "google" in row:
        print("  google    :", row["google"], f"  -> WER {row['google_wer']:.0%}")
    print()

RESULTS["speech"] = {"google_reachable": bool(google_ok), "clips": speech_rows}

Google Web Speech API not reachable from this machine: recognition connection failed: [Errno -2] Name or service not known 



[jfk] (11.0s)
  reference : and so my fellow americans ask not what your country can do for you ask what you can do for your country
  sphinx    : and i got my fellow americans ask not what your kind brain can do for you and when you can do for your country   -> WER 27%

[goforward] (2.8s)
  reference : go forward ten meters
  sphinx    : go forward ten meters   -> WER 0%

[english] (2.7s)
  reference : one two three
  sphinx    : one two three   -> WER 0%



### Why audio preprocessing matters
PocketSphinx expects 16 kHz mono audio. The JFK clip is recorded at a higher rate. Below, the same clip is
transcribed (a) after the careful resampling above and (b) by handing the original file straight to the library,
which does its own, cruder, rate conversion internally.

In [5]:
ref = CLIPS["jfk"][2]
proper = speech_rows[0]["sphinx"]
try:
    direct = transcribe(AUDIO / "jfk.flac", "sphinx")
except Exception as e:                        # e.g. no FLAC support on this machine
    direct = None; print("direct FLAC read not available:", e)

print("properly resampled :", proper, f"-> WER {wer(ref, proper):.0%}")
if direct is not None:
    print("original file as-is:", direct, f"-> WER {wer(ref, direct):.0%}")
    RESULTS["speech"]["jfk_direct_flac"] = {"hypothesis": direct, "wer": round(wer(ref, direct), 3)}

properly resampled : and i got my fellow americans ask not what your kind brain can do for you and when you can do for your country -> WER 27%
original file as-is: and i am i've gone back and not why y'all have drained into radio and what you can you or your country -> WER 73%


### Use it on your own recordings
* **A file:** `transcribe("my_recording.wav")` (WAV/AIFF/FLAC; run it through `prepare_wav` first if it is not 16 kHz mono).
* **The microphone:** the helper below needs the optional `PyAudio` package (`pip install pyaudio`), so it is
  defined here but not run.

In [6]:
def listen_and_transcribe(seconds=5, engine="sphinx"):
    """Record from the default microphone and transcribe (requires PyAudio)."""
    with sr.Microphone(sample_rate=16000) as source:
        recognizer.adjust_for_ambient_noise(source, duration=1)
        print("Speak now...")
        audio = recognizer.listen(source, phrase_time_limit=seconds)
    try:
        return (recognizer.recognize_sphinx(audio) if engine == "sphinx"
                else recognizer.recognize_google(audio))
    except sr.UnknownValueError:
        return ""

In [7]:
(ROOT / "results_speech.json").write_text(json.dumps(RESULTS, indent=2))
s = RESULTS["speech"]
print("Speech-to-text (Sphinx, offline) WER per clip: " + ", ".join(f"{r['clip']} {r['sphinx_wer']:.0%}" for r in s["clips"]))
print("\nSaved: results_speech.json")

Speech-to-text (Sphinx, offline) WER per clip: jfk 27%, goforward 0%, english 0%

Saved: results_speech.json
